<a href="https://colab.research.google.com/github/LeonimerMelo/Reinforcement-Learning/blob/Policy-Gradient/Introdu%C3%A7%C3%A3o_ao_DDPG_(Deep_Deterministic_Policy_Gradient)_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introdução ao DDPG (Deep Deterministic Policy Gradient)

O **DDPG (Deep Deterministic Policy Gradient)** é um algoritmo de **Aprendizado por Reforço Profundo (Deep Reinforcement Learning)** desenvolvido para resolver problemas em que as ações são **contínuas**.

Enquanto algoritmos como **DQN** escolhem ações discretas (esquerda, direita, cima, baixo), o DDPG consegue lidar com situações como:

* Controlar a velocidade de um veículo
* Ajustar o torque de um robô
* Controlar a posição de um braço robótico
* Gerenciar recursos em sistemas industriais

---

## Intuição Inicial

Imagine que você quer ensinar um carro autônomo a acelerar.

### DQN (ações discretas)

```text
Ação 1: Frear
Ação 2: Não fazer nada
Ação 3: Acelerar
```

### DDPG (ações contínuas)

```text
Aceleração = 0.37
Aceleração = -0.52
Aceleração = 0.91
```

O DDPG pode escolher qualquer valor dentro de um intervalo.

---

## Componentes Principais

O DDPG combina ideias de:

* Actor-Critic
* Deep Learning
* Replay Buffer
* Redes Alvo (Target Networks)

##Como Funciona o DDPG?
O Loop de Aprendizado:
- Observa o estado atual do ambiente
- Actor sugere uma ação baseada no estado
- Executa a ação e recebe recompensa e novo estado
- Armazena a experiência (estado, ação, recompensa, novo estado)
- Amostra um mini-batch de experiências da memória
- Atualiza o Critic para melhor avaliar ações
- Atualiza o Actor para escolher melhores ações
- Suaviza as redes alvo (Target Networks) para estabilidade

---

## 1. Actor

O **Actor** aprende a política:

$$
a = \pi(s)
$$

Recebe um estado e retorna uma ação.

Exemplo:

```python
estado = [velocidade, distância]
acao = actor.predict(estado)
```

Saída:

```python
0.72
```

Significando:

```text
Acelerar 72%
```

---

## 2. Critic

O **Critic** avalia se uma ação é boa.

Ele aproxima a função:

$$
Q(s,a)
$$

Exemplo:

```python
valor = critic.predict([estado, acao])
```

Saída:

```python
15.8
```

Quanto maior, melhor a ação.

---

# Arquitetura Geral

```text
         Estado
            │
            ▼
       +---------+
       | Actor   |
       +---------+
            │
            ▼
          Ação
            │
            ▼
       Ambiente
            │
      recompensa
            │
            ▼
       +---------+
       | Critic  |
       +---------+
```

---

# Rede Actor

Uma implementação simplificada usando TensorFlow/Keras:

```python
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

actor = Sequential([
    Dense(256, activation='relu', input_shape=(estado_dim,)),
    Dense(256, activation='relu'),
    Dense(acao_dim, activation='tanh')
])

actor.summary()
```

### Por que `tanh`?

Porque produz valores entre:

```text
[-1, 1]
```

Muito útil para controle contínuo.

---

# Rede Critic

O Critic recebe:

* Estado
* Ação

```python
from tensorflow.keras.layers import Input, Dense, Concatenate
from tensorflow.keras.models import Model

estado_input = Input(shape=(estado_dim,))
acao_input = Input(shape=(acao_dim,))

x = Concatenate()([estado_input, acao_input])

x = Dense(256, activation='relu')(x)
x = Dense(256, activation='relu')(x)

q_value = Dense(1)(x)

critic = Model(
    inputs=[estado_input, acao_input],
    outputs=q_value
)
```

---

# Replay Buffer

Treinar usando apenas a experiência atual gera alta correlação dos dados.

Por isso o DDPG armazena experiências:

```python
(s, a, r, s', done)
```

Exemplo:

```python
from collections import deque
import random

buffer = deque(maxlen=100000)

buffer.append(
    (estado,
     acao,
     recompensa,
     prox_estado,
     done)
)
```

Amostragem:

```python
batch = random.sample(buffer, 64)
```

---

# Target Networks

Um problema comum:

As redes mudam rapidamente durante o treinamento.

Para estabilizar:

* Target Actor
* Target Critic

```python
actor_target
critic_target
```

Elas são cópias lentas das redes principais.

---

## Atualização Suave (Soft Update)

$$
\theta' \leftarrow \tau\theta + (1-\tau)\theta'
$$

Normalmente:

```python
tau = 0.005
```

Código:

```python
for target, source in zip(
        target_model.weights,
        model.weights):

    target.assign(
        tau * source +
        (1 - tau) * target
    )
```

---

# Exploração

Se o Actor sempre executar sua melhor ação, ele não aprende coisas novas.

Adiciona-se ruído:

```python
acao = actor(estado)

acao += np.random.normal(
    0,
    0.1,
    size=acao_dim
)
```

Exemplo:

```text
Actor: 0.65
Ruído: 0.08

Ação final: 0.73
```

---

# Treinamento do Critic

O Critic aprende usando o alvo:

$$
y = r + \gamma Q'(s', \pi'(s'))
$$

onde:

* $(Q')$ = Target Critic
* $(\pi')$ = Target Actor

Código simplificado:

```python
next_actions = actor_target(next_states)

target_q = critic_target(
    [next_states, next_actions]
)

y = rewards + gamma * target_q
```

Treinamento:

```python
critic_loss = mse(
    y,
    critic([states, actions])
)
```

---

# Treinamento do Actor

O Actor quer maximizar:

$$
Q(s,\pi(s))
$$

Na prática:

```python
with tf.GradientTape() as tape:

    actions = actor(states)

    q_values = critic(
        [states, actions]
    )

    actor_loss = -tf.reduce_mean(
        q_values
    )
```

Observe o sinal negativo.

O objetivo é:

```text
Maximizar Q
=
Minimizar -Q
```

---

# Ciclo Completo do DDPG

```text
1. Observar estado

2. Actor gera ação

3. Adicionar ruído

4. Executar ação

5. Receber recompensa

6. Salvar experiência

7. Amostrar mini-batch

8. Atualizar Critic

9. Atualizar Actor

10. Atualizar Targets
```

---

# Pseudocódigo

```python
for episodio in range(num_episodios):

    estado = env.reset()

    while not done:

        acao = actor(estado)

        acao += ruido()

        prox_estado, recompensa, done, _ = env.step(acao)

        buffer.add(
            estado,
            acao,
            recompensa,
            prox_estado,
            done
        )

        train_critic()

        train_actor()

        update_targets()

        estado = prox_estado
```

---

# Exemplo usando Gymnasium

```python
import gymnasium as gym

env = gym.make(
    "Pendulum-v1"
)

estado, _ = env.reset()

for _ in range(100):

    acao = env.action_space.sample()

    estado, recompensa, terminated, truncated, _ = env.step(acao)

    if terminated or truncated:
        break
```

O ambiente **Pendulum-v1** é um dos exemplos clássicos para DDPG porque exige ações contínuas.

---

# Vantagens do DDPG

✅ Funciona com ações contínuas

✅ Mais eficiente que métodos puramente estocásticos

✅ Adequado para robótica e controle

✅ Usa Replay Buffer para reutilizar experiências

---

# Limitações

❌ Pode ser instável

❌ Sensível aos hiperparâmetros

❌ Exploração difícil

❌ Frequentemente superado por algoritmos mais modernos

* TD3
* SAC

---

# Resumo em uma frase

**DDPG é um algoritmo Actor-Critic off-policy que utiliza redes neurais para aprender políticas determinísticas em espaços de ações contínuas, empregando Replay Buffer e Target Networks para estabilizar o treinamento.**

Fluxo mental simplificado:

```text
Estado
   ↓
Actor
   ↓
Ação Contínua
   ↓
Ambiente
   ↓
Recompensa
   ↓
Critic avalia
   ↓
Atualiza Actor
   ↓
Repete
```

Essa é a base conceitual para compreender implementações mais modernas como TD3 e SAC, que surgiram justamente para corrigir algumas fragilidades do DDPG.


##DDPG e outros Algoritmos Policy Gradient

## 📊 Panorama Geral dos Algoritmos Policy Gradient

Para entender as diferenças, primeiro precisamos classificar os algoritmos:

```
POLICY GRADIENT METHODS
│
├── 🔵 ON-POLICY (Aprendem com a política atual)
│   ├── REINFORCE (Monte Carlo)
│   ├── A2C/A3C (Advantage Actor-Critic)
│   ├── PPO (Proximal Policy Optimization)
│   └── TRPO (Trust Region Policy Optimization)
│
└── 🔴 OFF-POLICY (Aprendem com experiências passadas)
    ├── DDPG (Deep Deterministic Policy Gradient)
    ├── TD3 (Twin Delayed DDPG)
    └── SAC (Soft Actor-Critic)
```

## 🎯 DDPG vs REINFORCE (Vanilla Policy Gradient)

| Característica | DDPG | REINFORCE |
|----------------|------|-----------|
| **Tipo** | Off-policy | On-policy |
| **Ações** | Contínuas | Discretas ou contínuas |
| **Usa rede neural?** | Sim (Actor-Critic) | Sim (Policy Network) |
| **Variance** | Baixa (usa Critic) | Alta (Monte Carlo) |
| **Sample Efficiency** | Alta (reutiliza experiências) | Baixa (descarta após usar) |
| **Estabilidade** | Mais estável (target networks) | Menos estável |

### Exemplo Conceitual:

```python
# REINFORCE - Atualização baseada em retorno completo
# Problema: Alta variância, lento para convergir
gradient = sum(R_t * ∇log π(a_t|s_t))  # R_t é o retorno total

# DDPG - Atualização baseada em Q-values
# Vantagem: Menor variância, mais eficiente
gradient = ∇Q(s, a) * ∇π(s)  # Usa o Critic para guiar
```

## 🔄 DDPG vs A2C/A3C (Advantage Actor-Critic)

| Característica | DDPG | A2C/A3C |
|----------------|------|---------|
| **Tipo** | Off-policy | On-policy |
| **Exploração** | Ruído (OU ou Gaussiano) | Entropia ou estocástica |
| **Política** | Determinística | Estocástica |
| **Buffer** | Experience Replay | Nenhum (ou pequeno) |
| **Paralelismo** | Não usa | Usa múltiplos workers (A3C) |
| **Eficiência** | Alta (replay buffer) | Média |

### Diferença na Política:

```python
# A2C/A3C - Política Estocástica
# Retorna probabilidades para cada ação
def policy(state):
    logits = network(state)
    return softmax(logits)  # Distribuição de probabilidade

# DDPG - Política Determinística
# Retorna uma ação específica
def policy(state):
    action = network(state)
    return action  # Valor contínuo específico
```

## 🛡️ DDPG vs PPO (Proximal Policy Optimization)

| Característica | DDPG | PPO |
|----------------|------|-----|
| **Tipo** | Off-policy | On-policy |
| **Ações** | Contínuas | Discretas ou contínuas |
| **Complexidade** | Moderada | Mais complexa |
| **Estabilidade** | Boa (com target nets) | Excelente (clipping) |
| **Sample Efficiency** | Alta | Média |
| **Tunagem** | Sensível a hiperparâmetros | Mais robusto |

### Diferença na Atualização:

```python
# PPO - Clipping para estabilidade
ratio = π(a|s) / π_old(a|s)
loss = -min(ratio * A, clip(ratio, 1-ε, 1+ε) * A)

# DDPG - Atualização direta do Critic
critic_loss = MSE(Q(s,a), target_Q)
actor_loss = -Q(s, π(s))
```

## 🆚 DDPG vs TD3 vs SAC (Comparação entre Off-Policy Contínuos)

| Característica | DDPG | TD3 | SAC |
|----------------|------|-----|-----|
| **Tipo** | Determinístico | Determinístico | Estocástico |
| **Número de Critics** | 1 | 2 (Twin) | 2 (Twin) |
| **Target Policy Smoothing** | ❌ | ✅ | ✅ |
| **Entropy Bonus** | ❌ | ❌ | ✅ |
| **Exploração** | Ruído externo | Ruído externo | Entropia interna |
| **Hiperparâmetros** | Muitos | Muitos | Mais robusto |
| **Performance** | Bom | Melhor | Melhor (geralmente) |

### Exemplo Comparativo:

```python
# DDPG - Atualização simples
q_target = r + γ * Q_target(s', π_target(s'))
actor_loss = -Q(s, π(s))

# TD3 - Melhorias sobre DDPG
# 1. Clipped Double-Q Learning
q_target = r + γ * min(Q1_target(s', a'), Q2_target(s', a'))

# 2. Target Policy Smoothing
a' = π_target(s') + noise  # Adiciona ruído à ação alvo

# SAC - Adiciona entropia
# Maximiza: E[Q(s,a)] + α * H(π(·|s))
actor_loss = α * log(π(a|s)) - Q(s, a)
```

## 📈 Análise Detalhada das Diferenças

### 1. **On-Policy vs Off-Policy**

```python
# ON-POLICY (REINFORCE, A2C, PPO)
# Só pode aprender com dados da política atual
for episode in episodes:
    trajectory = collect_data(policy)  # Coleta com política atual
    update(policy, trajectory)         # Atualiza
    # Descarta trajectory depois de usar

# OFF-POLICY (DDPG)
# Pode aprender com dados de políticas antigas
buffer = []
for step in steps:
    trajectory = collect_data(policy)  # Coleta com política atual
    buffer.add(trajectory)             # Guarda na memória
    batch = buffer.sample()            # Amostra aleatoriamente
    update(policy, batch)              # Atualiza com dados antigos
```

### 2. **Política Determinística vs Estocástica**

```python
# Política Estocástica (A2C, PPO)
# Mapeia estado → distribuição de probabilidade
def stochastic_policy(state):
    mean, std = network(state)
    return Normal(mean, std)  # Distribuição gaussiana

# Política Determinística (DDPG, TD3)
# Mapeia estado → ação específica
def deterministic_policy(state):
    action = network(state)
    return action  # Valor exato
```

### 3. **Exploração**

```python
# DDPG - Exploração via ruído externo
action = policy(state) + OU_noise()  # Ruído Ornstein-Uhlenbeck

# SAC - Exploração via entropia
# Política intrinsecamente estocástica
action = sample_from(policy_distribution)  # Amostra da distribuição

# PPO - Exploração via entropia
loss = -E[advantage] + β * H(π)  # Bonus de entropia
```

## 🎯 Quando Usar Cada Algoritmo?

### DDPG é melhor quando:
✅ **Ações contínuas** (ex: robótica, controle)

✅ **Dados são caros/limitados** (sample efficiency)

✅ **Ambiente é determinístico** (ou quase)

✅ **Você tem recursos computacionais** para replay buffer

### PPO é melhor quando:
✅ **Ações discretas ou contínuas** (versátil)

✅ **Estabilidade é crítica** (mais robusto)

✅ **Você quer um algoritmo "padrão"** (menos tunagem)

✅ **Paralelização é possível** (funciona bem com múltiplos envs)

### SAC é melhor quando:
✅ **Ações contínuas** (melhor que DDPG)

✅ **Exploração eficiente** é necessária

✅ **Hiperparâmetros são um problema** (mais robusto que DDPG)

✅ **Performance de ponta** é necessária

## 📊 Matriz de Decisão

```
                    AÇÕES CONTÍNUAS?
                          │
                 ┌───────┴───────┐
                SIM              NÃO
                 │                │
          ┌──────┴──────┐      DQN, A2C
          │             │
    OFF-POLICY?    ON-POLICY?
          │             │
     ┌────┴────┐   ┌────┴────┐
    SIM        NÃO SIM      NÃO
     │          │   │         │
   DDPG,      SAC  PPO     REINFORCE
   TD3
```

## 💡 Resumo Prático

| Algoritmo | Força | Fraqueza | Uso Principal |
|-----------|-------|----------|---------------|
| **DDPG** | Sample efficient, ações contínuas | Sensível, instável | Robótica básica |
| **TD3** | Mais estável que DDPG | Mais complexo | Robótica avançada |
| **SAC** | Muito robusto, explora bem | Computacionalmente pesado | Problemas complexos |
| **PPO** | Muito estável, versátil | Sample inefficiente | Aplicações reais |
| **A2C** | Simples, funciona bem | Não tão sample efficient | Jogos discretos |

## 🧪 Exemplo de Código: Mudança de Paradigma

```python
# DDPG - Treinamento Off-Policy
class DDPG:
    def train(self):
        batch = replay_buffer.sample(BATCH_SIZE)  # Reutiliza experiências
        # Atualiza usando dados de várias políticas diferentes
        self.update_actor(batch)
        self.update_critic(batch)

# PPO - Treinamento On-Policy  
class PPO:
    def train(self):
        trajectory = collect_episodes()  # Coleta nova experiência
        # Atualiza várias vezes com os MESMOS dados
        for _ in range(EPOCHS):
            self.update_policy(trajectory)  # Mas só pode usar esses dados
        # Descarta dados após usar
```

## 🎓 Conclusão

A principal diferença do DDPG está em sua natureza **off-policy determinística**. Enquanto outros algoritmos:

- **REINFORCE** é on-policy e sofre de alta variância
- **A2C/A3C** são on-policy estocásticos
- **PPO** é on-policy com clipping para estabilidade
- **SAC** é off-policy mas estocástico (com entropia)
- **TD3** é uma evolução do DDPG com melhorias de estabilidade

O DDPG ocupa um nicho específico: **aprendizado off-policy para ações contínuas**, oferecendo alta eficiência de amostragem em troca de maior sensibilidade a hiperparâmetros e menor estabilidade comparado a alternativas modernas como SAC.